# jaccode: train our own Jac model, end to end

Runs everything on a free Colab GPU:
1. Install the Jac compiler, Node and Ollama.
2. Check the benchmark against the real compiler.
3. Build verified training data with Claude as the teacher.
4. Fine-tune Qwen3.5-2B.
5. Score the trained model against the untrained one on the Jac benchmark.

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Add your Anthropic API key under 🔑 **Secrets** as `ANTHROPIC_API_KEY`, with notebook access on. It's only used to
  make training data. The finished model needs no key.
- Set `REPO` below to wherever you pushed jaccode.

**Cost:** the trial settings use roughly $10–30 of Claude usage (an estimate) and about 1–2 hours of free GPU.

In [ ]:
REPO = "https://github.com/YOUR_USER/jaccode"   # where jaccode is pushed
PROBLEMS = 50          # Python problems to translate into verified Jac (trial: 50; real run: 1000+)
AGENT_TASKS = 40       # agent tasks to record Claude solving (trial: 40; real run: 1000+)
BASE = "unsloth/Qwen3.5-2B"                       # model to fine-tune
BASE_GGUF = "hf.co/unsloth/Qwen3.5-2B-GGUF:Q5_K_M"  # same model, untrained, for the before/after comparison
HF_PUSH = ""           # optional: "your-name/jaccode-jac-2b-GGUF" to publish the result (log in below)
COMPARE_WITH_CLAUDE = False   # also score Claude on the benchmark (costs a few dollars)

import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

## 1. Install tools

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!curl -fsSL https://raw.githubusercontent.com/jaseci-labs/jaseci/main/scripts/install.sh | bash
!jac --version
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null
!node --version
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip install -q --upgrade unsloth unsloth_zoo

## 2. Get the code and check the benchmark against the real compiler

In [ ]:
%cd /content
!rm -rf jaccode jaseci
!git clone -q --depth 1 {REPO} jaccode
!git clone -q --depth 1 https://github.com/jaseci-labs/jaseci jaseci
%cd /content/jaccode
!npm install --no-audit --no-fund --silent && npm run build --silent
!pip install -q -r training/requirements.txt
!node dist/cli.js eval --reference --quiet --out /content/reference.json

In [ ]:
import json
ref = json.load(open("/content/reference.json"))["results"]
VALID = [r["id"] for r in ref if r["passed"]]
broken = [r["id"] for r in ref if not r["passed"]]
ONLY = ",".join(VALID)
print(f"{len(VALID)} benchmark tasks verified by the real compiler")
if broken:
    print("Reference solutions that need fixing (left out of scoring):", ", ".join(broken))
    for r in ref:
        if not r["passed"]:
            print(f"--- {r['id']}\n{(r.get('checkOutput') or '')[-800:]}\n{(r.get('testOutput') or '')[-800:]}")

## 3. Build verified training data

In [ ]:
%cd /content/jaccode/training
!python build_corpus.py --jaseci /content/jaseci --out data/corpus.jsonl --verify
!python translate.py --mbpp --limit {PROBLEMS} --out data/translated.jsonl --workers 4
!python make_agent_tasks.py --translated data/translated.jsonl --out data/agent_tasks
!node ../dist/cli.js eval --tasks data/agent_tasks --limit {AGENT_TASKS} --record data/runs.jsonl --quiet
!python prepare_sft.py --translated data/translated.jsonl --runs data/runs.jsonl --corpus data/corpus.jsonl \
    --general HuggingFaceTB/smoltalk --general-config all --out data/sft

## 4. Fine-tune (about 5 GB of GPU memory; a few minutes for the trial data)

In [ ]:
if HF_PUSH:
    from huggingface_hub import notebook_login
    notebook_login()
push = f"--push {HF_PUSH}" if HF_PUSH else ""
!python train_sft.py --data data/sft --out out/jac-2b --base {BASE} --quants q5_k_m {push}

## 5. Score it: untrained vs trained on the Jac benchmark

In [ ]:
import glob, subprocess, time, pathlib
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
gguf = sorted(glob.glob("out/jac-2b/gguf/*.gguf"), key=lambda p: pathlib.Path(p).stat().st_size)[-1]
# Unsloth writes a Modelfile with the chat template next to the GGUF; fall back to a bare one.
mf = pathlib.Path("out/jac-2b/gguf/Modelfile")
text = mf.read_text() if mf.exists() else "FROM placeholder\n"
lines = [l for l in text.splitlines() if not l.startswith("FROM")]
pathlib.Path("Modelfile").write_text("\n".join([f"FROM {gguf}", *lines]) + "\n")
print("model file:", gguf)
!ollama create jaccode-jac -f Modelfile
!ollama pull {BASE_GGUF}

In [ ]:
!node ../dist/cli.js eval -m ollama:{BASE_GGUF} --only {ONLY} --out /content/base.json --quiet
!node ../dist/cli.js eval -m ollama:jaccode-jac --only {ONLY} --out /content/trained.json --quiet
if COMPARE_WITH_CLAUDE:
    !node ../dist/cli.js eval --only {ONLY} --out /content/claude.json --quiet
files = "/content/base.json /content/trained.json" + (" /content/claude.json" if COMPARE_WITH_CLAUDE else "")
!python compare.py {files}

## 6. Keep the model

Download the GGUF below, or set `HF_PUSH` and re-run step 4. To use it on your computer with no API key:
```
ollama create jaccode-jac -f Modelfile     # Modelfile: FROM ./your-model.gguf
jaccode -m ollama:jaccode-jac
```
For a stronger model, raise `PROBLEMS` and `AGENT_TASKS`, re-run steps 3–5, and keep whichever version scores best.

In [ ]:
from google.colab import files
files.download(gguf)